#### Environment Setup

In [1]:
import os
import math
import json
import random
import gc
import warnings
import numpy as np
import pandas as pd
import torch

from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from transformers import (
    BertTokenizer,
    BertConfig,
    BertModel,
    BertForMaskedLM,
    DataCollatorForLanguageModeling,
    get_linear_schedule_with_warmup,
    set_seed,
)
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

#### Configuration

In [2]:
CSV_PATH = "discharge.csv"
TEXT_COLUMN = "text"
MODEL_NAME = "bionlp/bluebert_pubmed_mimic_uncased_L-12_H-768_A-12"
OUTPUT_DIR = "./blue_bert_results"

TEST_SIZE = 0.10
VAL_SIZE = 0.10
SEED = 42
RANDOM_STATE = 42

BLOCK_SIZE = 64
MLM_PROBABILITY = 0.15

PER_DEVICE_TRAIN_BATCH_SIZE = 16
PER_DEVICE_EVAL_BATCH_SIZE = 8
PER_DEVICE_EMBED_BATCH_SIZE = 32

NUM_TRAIN_EPOCHS = 1
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0

MAX_ROWS = 1000
SAVE_EMBEDDINGS = False
MAX_EMBEDDING_NOTES = 500
EMBED_MAX_LENGTH = 256

TOP_K_VALUES = [1, 3, 5]
BF16_REQUIRED = True

#### Device and Seed Setup

In [3]:
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for this notebook, but no GPU was found.")

if BF16_REQUIRED and not torch.cuda.is_bf16_supported():
    raise RuntimeError("BF16 was requested, but this GPU does not support BF16.")

device = torch.device("cuda:0")
AUTOCAST_DTYPE = torch.bfloat16

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

print("Device:", device)
print("GPU Name:", torch.cuda.get_device_name(0))
print("CUDA Available: Yes")
print("BF16 Enabled: Yes")

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("Seed set to:", SEED)

Device: cuda:0
GPU Name: NVIDIA GeForce RTX 4080 Laptop GPU
CUDA Available: Yes
BF16 Enabled: Yes
Seed set to: 42


#### Utility Functions

In [4]:
def clean_text(x):
    if pd.isna(x):
        return ""
    x = str(x)
    x = x.replace("\x00", " ")
    x = x.replace("\r", " ").replace("\n", " ")
    x = " ".join(x.split())
    return x.strip()


def compute_perplexity(loss_value):
    try:
        return float(math.exp(loss_value))
    except OverflowError:
        return float("inf")


def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    masked_embeddings = last_hidden_state * mask
    summed = masked_embeddings.sum(dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts


def clear_cuda():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()


def tokenize_and_chunk(dataset_dict, tokenizer, text_column="text", block_size=128):
    def tokenize_function(examples):
        return tokenizer(
            examples[text_column],
            add_special_tokens=True,
            truncation=False,
            return_attention_mask=True,
            return_special_tokens_mask=True
        )

    tokenized = dataset_dict.map(
        tokenize_function,
        batched=True,
        remove_columns=[text_column],
        desc="Tokenizing dataset"
    )

    def group_texts(examples):
        concatenated_examples = {}

        for k, v in examples.items():
            if len(v) == 0:
                continue
            if isinstance(v[0], list):
                concatenated_examples[k] = [item for sublist in v for item in sublist]

        total_length = len(concatenated_examples["input_ids"])
        total_length = (total_length // block_size) * block_size

        result = {
            k: [t[i:i + block_size] for i in range(0, total_length, block_size)]
            for k, t in concatenated_examples.items()
        }
        return result

    lm_dataset = tokenized.map(
        group_texts,
        batched=True,
        desc=f"Grouping into blocks of {block_size}"
    )

    return lm_dataset

#### Evaluation Functions

In [5]:
@torch.no_grad()
def evaluate_mlm_full(model, dataloader, device, topk_values=(1, 3, 5)):
    model.eval()

    total_loss = 0.0
    total_masked = 0
    total_correct = 0
    topk_correct = {k: 0 for k in topk_values}

    all_preds = []
    all_labels = []

    for batch in tqdm(dataloader, desc="Evaluating", leave=False):
        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
            outputs = model(**batch)

        loss = outputs.loss
        logits = outputs.logits
        labels = batch["labels"]

        valid_mask = labels != -100
        masked_count = valid_mask.sum().item()

        if masked_count == 0:
            continue

        masked_logits = logits[valid_mask]
        masked_labels = labels[valid_mask]

        preds = torch.argmax(masked_logits, dim=-1)

        total_loss += loss.item() * masked_count
        total_correct += (preds == masked_labels).sum().item()
        total_masked += masked_count

        all_preds.append(preds.detach().cpu())
        all_labels.append(masked_labels.detach().cpu())

        max_k = max(topk_values)
        topk_preds = torch.topk(masked_logits, k=max_k, dim=-1).indices

        for k in topk_values:
            correct_k = (topk_preds[:, :k] == masked_labels.unsqueeze(1)).any(dim=1).sum().item()
            topk_correct[k] += correct_k

        del outputs, logits, labels, valid_mask, masked_logits, masked_labels, preds, topk_preds

    avg_loss = total_loss / max(total_masked, 1)
    accuracy = total_correct / max(total_masked, 1)

    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()

    micro_f1 = f1_score(all_labels, all_preds, average="micro")
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)

    metrics = {
        "loss": round(float(avg_loss), 4),
        "accuracy": round(float(accuracy), 4),
        "micro_f1": round(float(micro_f1), 4),
        "macro_f1": round(float(macro_f1), 4),
    }

    for k in topk_values:
        metrics[f"precision@{k}"] = round(float(topk_correct[k] / max(total_masked, 1)), 4)

    return metrics


@torch.no_grad()
def extract_embeddings(texts, tokenizer, model, device, batch_size=32, max_length=256):
    model.eval()
    all_embeddings = []

    for i in tqdm(range(0, len(texts), batch_size), desc="Extracting embeddings"):
        batch_texts = texts[i:i + batch_size]

        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        encoded = {k: v.to(device, non_blocking=True) for k, v in encoded.items()}

        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
            outputs = model(**encoded)
            pooled = mean_pool(outputs.last_hidden_state, encoded["attention_mask"])

        all_embeddings.append(pooled.detach().cpu())

    return torch.cat(all_embeddings, dim=0).numpy()

#### Data Loading

In [6]:
df = pd.read_csv(CSV_PATH)

required_columns = [
    "note_id",
    "subject_id",
    "hadm_id",
    "note_type",
    "note_seq",
    "charttime",
    "storetime",
    TEXT_COLUMN
]

missing_cols = [col for col in required_columns if col not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

df[TEXT_COLUMN] = df[TEXT_COLUMN].apply(clean_text)
df = df[df[TEXT_COLUMN].str.len() > 0].reset_index(drop=True)

if MAX_ROWS is not None:
    df = df.iloc[:MAX_ROWS].copy()

print("Dataset shape after cleaning:", df.shape)
print()
print(df[TEXT_COLUMN].iloc[0][:1000])

Dataset shape after cleaning: (1000, 8)

Name: ___ Unit No: ___ Admission Date: ___ Discharge Date: ___ Date of Birth: ___ Sex: F Service: MEDICINE Allergies: No Known Allergies / Adverse Drug Reactions Attending: ___ Chief Complaint: Worsening ABD distension and pain Major Surgical or Invasive Procedure: Paracentesis History of Present Illness: ___ HCV cirrhosis c/b ascites, hiv on ART, h/o IVDU, COPD, bioplar, PTSD, presented from OSH ED with worsening abd distension over past week. Pt reports self-discontinuing lasix and spirnolactone ___ weeks ago, because she feels like "they don't do anything" and that she "doesn't want to put more chemicals in her." She does not follow Na-restricted diets. In the past week, she notes that she has been having worsening abd distension and discomfort. She denies ___ edema, or SOB, or orthopnea. She denies f/c/n/v, d/c, dysuria. She had food poisoning a week ago from eating stale cake (n/v 20 min after food ingestion), which resolved the same day. S

#### Data Splitting

In [7]:
train_df, test_df = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    shuffle=True
)

val_relative = VAL_SIZE / (1 - TEST_SIZE)

train_df, val_df = train_test_split(
    train_df,
    test_size=val_relative,
    random_state=RANDOM_STATE,
    shuffle=True
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train size      :", len(train_df))
print("Validation size :", len(val_df))
print("Test size       :", len(test_df))

Train size      : 800
Validation size : 100
Test size       : 100


#### Dataset Conversion

In [8]:
dataset_dict = DatasetDict({
    "train": Dataset.from_pandas(train_df[[TEXT_COLUMN]], preserve_index=False),
    "validation": Dataset.from_pandas(val_df[[TEXT_COLUMN]], preserve_index=False),
    "test": Dataset.from_pandas(test_df[[TEXT_COLUMN]], preserve_index=False),
})

dataset_dict

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 800
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 100
    })
    test: Dataset({
        features: ['text'],
        num_rows: 100
    })
})

#### Tokenizer and Model Loading

In [9]:
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

config = BertConfig.from_pretrained(MODEL_NAME)
mlm_model = BertForMaskedLM.from_pretrained(MODEL_NAME, config=config).to(device)
embedding_model = BertModel.from_pretrained(MODEL_NAME, config=config).to(device)

print("Tokenizer loaded successfully")
print("Config loaded successfully")
print("Model loaded successfully")
print("Model name:", MODEL_NAME)

Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bionlp/bluebert_pubmed_mimic_uncased_L-12_H-768_A-12
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.pooler.dense.weight     | UNEXPECTED |  | 
bert.embeddings.position_ids | UNEXPECTED |  | 
bert.pooler.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight  | UNEXPECTED |  | 
cls.seq_relationship.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bionlp/bluebert_pubmed_mimic_uncased_L-12_H-768_A-12
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Tokenizer loaded successfully
Config loaded successfully
Model loaded successfully
Model name: bionlp/bluebert_pubmed_mimic_uncased_L-12_H-768_A-12


#### Tokenization and Chunking

In [10]:
lm_dataset = tokenize_and_chunk(
    dataset_dict=dataset_dict,
    tokenizer=tokenizer,
    text_column=TEXT_COLUMN,
    block_size=BLOCK_SIZE
)

print("Train blocks      :", len(lm_dataset["train"]))
print("Validation blocks :", len(lm_dataset["validation"]))
print("Test blocks       :", len(lm_dataset["test"]))

Tokenizing dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

Tokenizing dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Tokenizing dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Grouping into blocks of 64:   0%|          | 0/800 [00:00<?, ? examples/s]

Grouping into blocks of 64:   0%|          | 0/100 [00:00<?, ? examples/s]

Grouping into blocks of 64:   0%|          | 0/100 [00:00<?, ? examples/s]

Train blocks      : 37628
Validation blocks : 4962
Test blocks       : 4413


#### Data Collator

In [11]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=MLM_PROBABILITY
)

print("Data collator ready")

Data collator ready


#### Data Loaders

In [12]:
def collate_cuda(examples):
    batch = data_collator(examples)
    batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
    return batch

train_loader = DataLoader(
    lm_dataset["train"],
    batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_cuda,
    num_workers=0,
    pin_memory=False
)

val_loader = DataLoader(
    lm_dataset["validation"],
    batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_cuda,
    num_workers=0,
    pin_memory=False
)

test_loader = DataLoader(
    lm_dataset["test"],
    batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_cuda,
    num_workers=0,
    pin_memory=False
)

print("DataLoaders created successfully")
print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

DataLoaders created successfully
Train batches: 2352
Validation batches: 621
Test batches: 552


#### Optimizer and Scheduler

In [13]:
optimizer = torch.optim.AdamW(
    mlm_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

num_training_steps = NUM_TRAIN_EPOCHS * len(train_loader)
num_warmup_steps = int(0.1 * num_training_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps
)

print("Optimizer and scheduler ready")
print("Training steps:", num_training_steps)
print("Warmup steps:", num_warmup_steps)

Optimizer and scheduler ready
Training steps: 2352
Warmup steps: 235


#### Model Training

In [14]:
mlm_model.train()
global_step = 0
running_loss = 0.0
running_masked = 0

for epoch in range(NUM_TRAIN_EPOCHS):
    progress_bar = tqdm(train_loader, desc=f"Training Epoch {epoch + 1}")

    for batch in progress_bar:
        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
            outputs = mlm_model(**batch)
            loss = outputs.loss
            labels = batch["labels"]
            masked_tokens = (labels != -100).sum().item()

        loss.backward()
        torch.nn.utils.clip_grad_norm_(mlm_model.parameters(), MAX_GRAD_NORM)
        optimizer.step()
        scheduler.step()

        global_step += 1
        running_loss += loss.item() * masked_tokens
        running_masked += masked_tokens

        progress_bar.set_postfix({
            "step": global_step,
            "loss": round(loss.item(), 4)
        })

train_loss = running_loss / max(running_masked, 1)

print("Training completed")
print({"train_loss": round(train_loss, 4)})

Training Epoch 1:   0%|          | 0/2352 [00:00<?, ?it/s]

Training completed
{'train_loss': 0.9691}


#### Model Evaluation

In [15]:
clear_cuda()
train_metrics = evaluate_mlm_full(mlm_model, train_loader, device, TOP_K_VALUES)

clear_cuda()
val_metrics = evaluate_mlm_full(mlm_model, val_loader, device, TOP_K_VALUES)

clear_cuda()
test_metrics = evaluate_mlm_full(mlm_model, test_loader, device, TOP_K_VALUES)

print("Train metrics:")
print(train_metrics)

print("Validation metrics:")
print(val_metrics)

print("Test metrics:")
print(test_metrics)

Evaluating:   0%|          | 0/2352 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/621 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/552 [00:00<?, ?it/s]

Train metrics:
{'loss': 0.7719, 'accuracy': 0.8213, 'micro_f1': 0.8213, 'macro_f1': 0.5851, 'precision@1': 0.8214, 'precision@3': 0.9005, 'precision@5': 0.9231}
Validation metrics:
{'loss': 0.7877, 'accuracy': 0.8185, 'micro_f1': 0.8185, 'macro_f1': 0.6314, 'precision@1': 0.8187, 'precision@3': 0.8962, 'precision@5': 0.919}
Test metrics:
{'loss': 0.7638, 'accuracy': 0.8227, 'micro_f1': 0.8227, 'macro_f1': 0.6429, 'precision@1': 0.8227, 'precision@3': 0.9028, 'precision@5': 0.9245}


#### Final Metrics Dictionary

In [16]:
final_results = {
    "Training Loss": train_metrics["loss"],
    "Training Accuracy": train_metrics["accuracy"],
    "Training Micro F1": train_metrics["micro_f1"],
    "Training Macro F1": train_metrics["macro_f1"],
    "Training Precision@1": train_metrics["precision@1"],
    "Training Precision@3": train_metrics["precision@3"],
    "Training Precision@5": train_metrics["precision@5"],

    "Validation Loss": val_metrics["loss"],
    "Validation Accuracy": val_metrics["accuracy"],
    "Validation Micro F1": val_metrics["micro_f1"],
    "Validation Macro F1": val_metrics["macro_f1"],
    "Validation Precision@1": val_metrics["precision@1"],
    "Validation Precision@3": val_metrics["precision@3"],
    "Validation Precision@5": val_metrics["precision@5"],

    "Testing Loss": test_metrics["loss"],
    "Testing Accuracy": test_metrics["accuracy"],
    "Testing Micro F1": test_metrics["micro_f1"],
    "Testing Macro F1": test_metrics["macro_f1"],
    "Testing Precision@1": test_metrics["precision@1"],
    "Testing Precision@3": test_metrics["precision@3"],
    "Testing Precision@5": test_metrics["precision@5"],
}

#### Results Table

In [17]:
results_df = pd.DataFrame({
    "Metric": [
        "Loss",
        "Accuracy",
        "Micro F1",
        "Macro F1",
        "Precision@1",
        "Precision@3",
        "Precision@5"
    ],
    "Training": [
        train_metrics["loss"],
        train_metrics["accuracy"],
        train_metrics["micro_f1"],
        train_metrics["macro_f1"],
        train_metrics["precision@1"],
        train_metrics["precision@3"],
        train_metrics["precision@5"],
    ],
    "Validation": [
        val_metrics["loss"],
        val_metrics["accuracy"],
        val_metrics["micro_f1"],
        val_metrics["macro_f1"],
        val_metrics["precision@1"],
        val_metrics["precision@3"],
        val_metrics["precision@5"],
    ],
    "Testing": [
        test_metrics["loss"],
        test_metrics["accuracy"],
        test_metrics["micro_f1"],
        test_metrics["macro_f1"],
        test_metrics["precision@1"],
        test_metrics["precision@3"],
        test_metrics["precision@5"],
    ]
})

print("=" * 80)
print("FINAL MODEL PERFORMANCE METRICS")
print("=" * 80)
display(results_df)

FINAL MODEL PERFORMANCE METRICS


,Metric,Training,Validation,Testing
0,Loss,0.7719,0.7877,0.7638
1,Accuracy,0.8213,0.8185,0.8227
2,Micro F1,0.8213,0.8185,0.8227
3,Macro F1,0.5851,0.6314,0.6429
4,Precision@1,0.8214,0.8187,0.8227
5,Precision@3,0.9005,0.8962,0.9028
6,Precision@5,0.9231,0.9190,0.9245


#### Save Results

In [18]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

json_path = os.path.join(OUTPUT_DIR, "final_metrics.json")
csv_path = os.path.join(OUTPUT_DIR, "final_metrics.csv")

with open(json_path, "w") as f:
    json.dump(final_results, f, indent=4)

results_df.to_csv(csv_path, index=False)

print("Saved JSON to:", json_path)
print("Saved CSV to :", csv_path)

Saved JSON to: ./blue_bert_results\final_metrics.json
Saved CSV to : ./blue_bert_results\final_metrics.csv


#### Perplexity

In [19]:
perplexity_results = {
    "Train Perplexity": round(compute_perplexity(train_metrics["loss"]), 4),
    "Validation Perplexity": round(compute_perplexity(val_metrics["loss"]), 4),
    "Test Perplexity": round(compute_perplexity(test_metrics["loss"]), 4),
}

perplexity_df = pd.DataFrame([perplexity_results])

print("=" * 80)
print("PERPLEXITY RESULTS")
print("=" * 80)
display(perplexity_df)

PERPLEXITY RESULTS


,Train Perplexity,Validation Perplexity,Test Perplexity
0,2.1639,2.1983,2.1464


#### Final Summary Table

In [20]:
summary_df = pd.DataFrame({
    "Split": ["Training", "Validation", "Testing"],
    "Loss": [
        final_results["Training Loss"],
        final_results["Validation Loss"],
        final_results["Testing Loss"]
    ],
    "Accuracy": [
        final_results["Training Accuracy"],
        final_results["Validation Accuracy"],
        final_results["Testing Accuracy"]
    ],
    "Micro F1": [
        final_results["Training Micro F1"],
        final_results["Validation Micro F1"],
        final_results["Testing Micro F1"]
    ],
    "Macro F1": [
        final_results["Training Macro F1"],
        final_results["Validation Macro F1"],
        final_results["Testing Macro F1"]
    ],
    "Precision@1": [
        final_results["Training Precision@1"],
        final_results["Validation Precision@1"],
        final_results["Testing Precision@1"]
    ],
    "Precision@3": [
        final_results["Training Precision@3"],
        final_results["Validation Precision@3"],
        final_results["Testing Precision@3"]
    ],
    "Precision@5": [
        final_results["Training Precision@5"],
        final_results["Validation Precision@5"],
        final_results["Testing Precision@5"]
    ]
})

print("=" * 80)
print("FINAL SUMMARY")
print("=" * 80)
display(summary_df)

FINAL SUMMARY


,Split,Loss,Accuracy,Micro F1,Macro F1,Precision@1,Precision@3,Precision@5
0,Training,0.7719,0.8213,0.8213,0.5851,0.8214,0.9005,0.9231
1,Validation,0.7877,0.8185,0.8185,0.6314,0.8187,0.8962,0.9190
2,Testing,0.7638,0.8227,0.8227,0.6429,0.8227,0.9028,0.9245
